In [17]:
import SimpleITK as sitk
import numpy as np
import os
import torch
from torch.nn.functional import one_hot


dir = "/mnt/raid/C1_ML_Analysis/simulated_data_export/placenta_simu/"

In [ ]:
###################### TEST ONE HOT ######################
array = one_hot(torch.tensor(sitk.GetArrayFromImage(sitk.ReadImage("/mnt/raid/C1_ML_Analysis/simulated_data_export/placenta_simu/IB3_label11/M_label/120.nrrd"))).long().unsqueeze(0)).permute(0,3,1,2)

print(array.shape)

In [15]:
def get_frames(parent_dir):
    parent_folders = []
    folders = []
    valid_frames = {}
    for parent_folder in sorted(os.listdir(parent_dir)):
        parent_folder_path = os.path.join(parent_dir,parent_folder)
        parent_folders.append(parent_folder_path)
        for folder in sorted(os.listdir(parent_folder_path)):
            if ".nrrd" not in folder:
                folder_path = os.path.join(parent_folder_path,folder)
                folders.append(folder_path)

    label_folders = []
    for path in folders:
        if "_label" in path.split("/")[-1]:
            label_folders.append(path)

    for file in label_folders:
        frames = []
        for frame in os.listdir(file):
            img_path = os.path.join(file,frame)
            array = sitk.GetArrayFromImage(sitk.ReadImage(img_path))
            arg = np.argwhere(array==7)
            frames.append(arg)
        valid_frames[file.replace("label","bmap_guided").replace("bmap_guided11","label11")] = [i for i, lst in enumerate(frames) if lst.all() and lst.size > 0]

    return valid_frames

In [ ]:
dict = get_frames(dir)
print(dict)

In [17]:
import json

# Save as a JSON file
with open("dictionary_bmap.json", "w") as f:
    json.dump(dict, f)


In [ ]:
# Load JSON file
with open("dictionary_diffusor.json", "r") as f:
    loaded_data = json.load(f)

print(type(loaded_data)) # Output: {'name': 'Alice', 'age': 25, 'city': 'New York'}


In [ ]:
import os
import numpy as np
import nrrd
import json

image_dir = '/mnt/raid/home/ajarry/data/image_capture_output'
images=[]
files = [f for f in os.listdir(image_dir) if not f.startswith('.')]


sorted_files = sorted(files,key=lambda x: int(os.path.splitext(x)[0]))

for file in sorted_files:
    path = os.path.join(image_dir,file)
    with open (path) as f:
        dico = json.load(f)
        images.append(dico['image'])

volume = np.stack(images,axis=2)
print(volume.shape)
out = image_dir + "/fake.nrrd"
nrrd.write(out,volume)



In [ ]:

import plotly.express as px

data = sitk.GetArrayFromImage(sitk.ReadImage('/mnt/raid/home/ajarry/data/image_capture_output/fake.nrrd'))
print(data.shape)
fig = px.imshow(data,animation_frame=0, binary_string=True)
fig.show()
